#  Convolutional NN.  Learns for the smaller patterns rather than global pattern.  (Eg. three horizontal lines)

![alt text](localpattern.png "local pattern")

PyTorch version of `cnn.ipynb` (which used TensorFlow / Keras).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
from torchinfo import summary

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
print ("Checking any GPU is available")
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

In [ ]:
# hand written digits, 28 X 28 images
# torchvision downloads MNIST into the ./data folder the first time
mnist_train = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)

In [ ]:
# .data is a uint8 tensor of shape (N, 28, 28); .targets holds the digit 0-9
train_images = mnist_train.data.numpy()
train_labels = mnist_train.targets.numpy()
test_images = mnist_test.data.numpy()
test_labels = mnist_test.targets.numpy()

printable_train_images = train_images
printable_test_images = test_images

print(train_images.shape)

In [ ]:
print(train_images[50])  #kinda looks like a 3

**Note**: PyTorch puts the color channel *first*: (N, channels, rows, cols).
Keras put it last: (N, rows, cols, channels).

In [ ]:
train_images = train_images.reshape((60000,1,28,28))  # add a channel dimension
print(train_images.shape) # 60000 images, 1 color, 28 X, 28 Y

In [ ]:
#print(train_images[50])  # take a look at the format

## normalize to a value between 0 and 1

In [ ]:
train_images = train_images.astype('float32')/255  #another way of normalizing

In [ ]:
#print(train_images[50])  # another sanity check

In [ ]:
test_images = test_images.reshape(10000,1,28,28)
test_images = test_images.astype('float32')/255

## Labels (no one hot encode needed)
Keras' `categorical_crossentropy` wanted one-hot rows like `[0,0,0,1,0,0,0,0,0,0]`.

PyTorch's `nn.CrossEntropyLoss` wants the class number instead (e.g. `3`), so the labels stay as they are.

In [ ]:
# ---------------------------------------------------------
# Convert numpy data to PyTorch tensors
# ---------------------------------------------------------

train_images = torch.tensor(train_images)
test_images = torch.tensor(test_images)

# CrossEntropyLoss wants class numbers as int64 (long)
train_labels = torch.tensor(train_labels, dtype=torch.long)
test_labels = torch.tensor(test_labels, dtype=torch.long)

print(train_images.shape, train_labels.shape)
print(test_images.shape, test_labels.shape)

# For comparison, this is what the Keras one-hot version looked like:
print(nn.functional.one_hot(train_labels[:3], num_classes=10))

In [ ]:
# ---------------------------------------------------------
# Create batches
# ---------------------------------------------------------

train_loader = DataLoader(
    TensorDataset(train_images, train_labels),
    batch_size=2500,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(test_images, test_labels),
    batch_size=2500,
    shuffle=False
)

# How a convolution works (toy example)

Before building the real network, let's do it by hand on a tiny 9 X 9 picture of an **X**.
Pixels are **1** (white) or **-1** (black).

A **filter** (also called a *kernel*) is a small 3 X 3 grid of weights that describes one little pattern,
for example a short diagonal line `\`.

**Color key for every plot below:** the input picture is black/white.
Filter weights and computed values use **red = positive**, **blue = negative**, **white = zero**.

In [ ]:
# ---------------------------------------------------------
# Helpers for drawing small grids of numbers
# ---------------------------------------------------------
import matplotlib.patches as patches
import torch.nn.functional as F

# Colors for the layer types (used in the pipeline diagrams)
LAYER_COLORS = {
    "Conv2d": "#2a78d6",     # blue
    "ReLU": "#eb6834",       # orange
    "MaxPool2d": "#1baf7a",  # aqua
    "Flatten": "#eda100",    # yellow
    "Linear": "#e87ba4",     # magenta
}

signed_cmap = plt.get_cmap("RdBu_r").copy()
signed_cmap.set_bad("whitesmoke")   # cells not computed yet (NaN) show light gray


def show_grid(ax, grid, title="", signed=True, annotate=True, fmt="{:.0f}", limit=None):
    """Draw a 2-D array as colored squares, optionally writing each number in its square."""
    grid = np.asarray(grid, dtype=float)

    if signed:
        if limit is None:
            limit = np.nanmax(np.abs(grid)) or 1
        ax.imshow(grid, cmap=signed_cmap, vmin=-limit, vmax=limit)
    else:
        limit = 1
        ax.imshow(grid, cmap="gray", vmin=-1, vmax=1)

    if annotate:
        for (r, c), v in np.ndenumerate(grid):
            if np.isnan(v):
                continue
            # pick a text color that is readable on the square's color
            dark_square = (not signed and v < 0) or (signed and abs(v) > 0.6 * limit)
            text = fmt.format(v)
            if float(text) == 0:
                text = fmt.format(0)    # show 0.0 instead of -0.0
            ax.text(c, r, text, ha="center", va="center", fontsize=9,
                    color="white" if dark_square else "black")

    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def add_box(ax, row, col, size=3, color="#eda100", width=3):
    """Draw a square outline over cells (row..row+size-1, col..col+size-1)."""
    ax.add_patch(patches.Rectangle((col - 0.5, row - 0.5), size, size,
                                   fill=False, edgecolor=color, linewidth=width))

In [ ]:
# ---------------------------------------------------------
# A 9 X 9 picture of an X, and three 3 X 3 filters
# ---------------------------------------------------------

toy_image = np.array([
    [-1, -1, -1, -1, -1, -1, -1, -1, -1],
    [-1,  1, -1, -1, -1, -1, -1,  1, -1],
    [-1, -1,  1, -1, -1, -1,  1, -1, -1],
    [-1, -1, -1,  1, -1,  1, -1, -1, -1],
    [-1, -1, -1, -1,  1, -1, -1, -1, -1],
    [-1, -1, -1,  1, -1,  1, -1, -1, -1],
    [-1, -1,  1, -1, -1, -1,  1, -1, -1],
    [-1,  1, -1, -1, -1, -1, -1,  1, -1],
    [-1, -1, -1, -1, -1, -1, -1, -1, -1],
], dtype=np.float32)

toy_filters = {
    "diagonal \\": np.array([[ 1, -1, -1],
                              [-1,  1, -1],
                              [-1, -1,  1]], dtype=np.float32),
    "center X":     np.array([[ 1, -1,  1],
                              [-1,  1, -1],
                              [ 1, -1,  1]], dtype=np.float32),
    "diagonal /":   np.array([[-1, -1,  1],
                              [-1,  1, -1],
                              [ 1, -1, -1]], dtype=np.float32),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1, 1, 1]})
show_grid(axes[0], toy_image, "Input image (9 X 9)", signed=False)
for ax, (name, k) in zip(axes[1:], toy_filters.items()):
    show_grid(ax, k, f"Filter: {name}", limit=1)
plt.tight_layout()
plt.show()

## One step of a convolution

Place the filter on a 3 X 3 patch of the image, **multiply** each pixel by the filter weight on top of it,
then **add up** the 9 products. (Here we also divide by 9 so the answer is between -1 and 1.)

* Every pixel agrees with the filter  ->  big positive number: *"the pattern is here!"*
* Pixels disagree with the filter  ->  small or negative number: *"not here"*

In [ ]:
def show_one_step(image, kernel, row, col, kernel_name=""):
    """Show image patch  X  filter  =  products  ->  sum, for one filter position."""
    patch = image[row:row + 3, col:col + 3]
    products = patch * kernel
    value = products.sum() / kernel.size

    fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), gridspec_kw={"width_ratios": [2.2, 1, 1, 1, 1.1]})

    show_grid(axes[0], image, f"Image, patch at row {row}, col {col}", signed=False, annotate=False)
    add_box(axes[0], row, col)

    show_grid(axes[1], patch, "Patch", signed=False)
    show_grid(axes[2], kernel, f"X  filter {kernel_name}", limit=1)
    show_grid(axes[3], products, "=  products", limit=1)

    axes[4].axis("off")
    axes[4].text(0.0, 0.5,
                 f"sum = {products.sum():.0f}\n\n{products.sum():.0f} / 9 = {value:.2f}",
                 fontsize=15, va="center")

    plt.tight_layout()
    plt.show()
    return value


diag = toy_filters["diagonal \\"]

# Perfect match: the patch looks exactly like the filter
show_one_step(toy_image, diag, row=1, col=1, kernel_name="\\")

# Poor match: the patch is the OTHER diagonal
show_one_step(toy_image, diag, row=1, col=5, kernel_name="\\")

## Sliding the filter over the whole image

Repeat that step at **every** position (left to right, top to bottom). Each answer goes into one square of a new grid
called a **feature map**. It is a map of *where* the pattern was found.

A 3 X 3 filter fits in only 7 positions across a 9-pixel row, so the 9 X 9 image becomes a 7 X 7 feature map
(this is the "lost border" mentioned in the model section).

In [ ]:
def convolve(image, kernel):
    """A convolution written with plain loops (no padding, stride 1)."""
    k = kernel.shape[0]
    out_size = image.shape[0] - k + 1
    out = np.zeros((out_size, out_size), dtype=np.float32)

    for r in range(out_size):
        for c in range(out_size):
            out[r, c] = (image[r:r + k, c:c + k] * kernel).sum() / kernel.size

    return out


# Show the feature map filling in as the filter slides
steps_to_show = [(0, 0), (0, 1), (0, 6), (1, 1), (3, 3), (6, 6)]
full_map = convolve(toy_image, diag)

fig, axes = plt.subplots(2, len(steps_to_show), figsize=(18, 6))

for i, (r, c) in enumerate(steps_to_show):
    show_grid(axes[0, i], toy_image, f"Filter at ({r}, {c})", signed=False, annotate=False)
    add_box(axes[0, i], r, c)

    # Everything up to (and including) this position has been computed
    partial = np.full_like(full_map, np.nan)
    done = r * full_map.shape[1] + c + 1
    partial.flat[:done] = full_map.flat[:done]

    show_grid(axes[1, i], partial, "Feature map so far", fmt="{:.2f}", limit=1)
    add_box(axes[1, i], r, c, size=1)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# PyTorch does exactly the same thing with F.conv2d
# ---------------------------------------------------------

# PyTorch wants (batch, channels, rows, cols) for the image
# and (filters, channels, rows, cols) for the weights
image_t = torch.tensor(toy_image).reshape(1, 1, 9, 9)
filters_t = torch.tensor(np.stack(list(toy_filters.values()))).unsqueeze(1)   # (3, 1, 3, 3)

toy_feature_maps = F.conv2d(image_t, filters_t) / 9    # (1, 3, 7, 7): one map per filter

print("image   ", tuple(image_t.shape))
print("filters ", tuple(filters_t.shape))
print("output  ", tuple(toy_feature_maps.shape))
print("Same as our loop version?", np.allclose(toy_feature_maps[0, 0].numpy(), full_map))

fig, axes = plt.subplots(3, 3, figsize=(12, 13), gridspec_kw={"width_ratios": [1.3, 0.6, 1]})

for i, name in enumerate(toy_filters):
    show_grid(axes[i, 0], toy_image, "Image", signed=False, annotate=False)
    show_grid(axes[i, 1], toy_filters[name], f"X  {name}", limit=1)
    show_grid(axes[i, 2], toy_feature_maps[0, i].numpy(), "=  feature map", fmt="{:.2f}", limit=1)

plt.tight_layout()
plt.show()

# Notice: each feature map is brightest (red) exactly where its pattern sits in the X.

## ReLU: keep the positives, zero out the negatives

`ReLU(x) = max(0, x)`. Negative values ("definitely not this pattern") become 0, positive values pass through unchanged.

In [ ]:
toy_relu = torch.relu(toy_feature_maps)      # same as nn.ReLU()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1.2, 1.2]})

x = np.linspace(-1, 1, 100)
axes[0].plot(x, np.maximum(0, x), color=LAYER_COLORS["ReLU"], linewidth=2)
axes[0].axhline(0, color="lightgray", linewidth=1, zorder=0)
axes[0].axvline(0, color="lightgray", linewidth=1, zorder=0)
axes[0].set_title("ReLU(x) = max(0, x)")
axes[0].set_xlabel("input")
axes[0].set_ylabel("output")

show_grid(axes[1], toy_feature_maps[0, 0].numpy(), "Feature map (diagonal \\)", fmt="{:.2f}", limit=1)
show_grid(axes[2], toy_relu[0, 0].numpy(), "After ReLU", fmt="{:.2f}", limit=1)

plt.tight_layout()
plt.show()

## Max pooling: shrink the map, keep the strongest signal

Cut the map into 2 X 2 windows (moving 2 squares at a time) and keep only the **largest** value in each window.
The map gets 4 times smaller, but the "pattern found here" signals survive.
A 7 X 7 map has an odd leftover row/column, which is dropped, giving 3 X 3.

In [ ]:
relu_map = toy_relu[0, 0].numpy()
pooled = F.max_pool2d(toy_relu, kernel_size=2, stride=2)    # same as nn.MaxPool2d(2, 2)
pooled_map = pooled[0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), gridspec_kw={"width_ratios": [7, 3]})

show_grid(axes[0], relu_map, "After ReLU (7 X 7), 2 X 2 windows", fmt="{:.2f}", limit=1)
show_grid(axes[1], pooled_map, "After max pool (3 X 3)", fmt="{:.2f}", limit=1)

for pr in range(pooled_map.shape[0]):
    for pc in range(pooled_map.shape[1]):
        add_box(axes[0], 2 * pr, 2 * pc, size=2, color=LAYER_COLORS["MaxPool2d"], width=2)

        # circle the winner in each window
        window = relu_map[2 * pr:2 * pr + 2, 2 * pc:2 * pc + 2]
        wr, wc = np.unravel_index(window.argmax(), window.shape)
        axes[0].add_patch(patches.Circle((2 * pc + wc, 2 * pr + wr), 0.42, fill=False,
                                         edgecolor="black", linewidth=2))

# gray out the dropped last row and column
axes[0].add_patch(patches.Rectangle((-0.5, 5.5), 7, 1, color="gray", alpha=0.35))
axes[0].add_patch(patches.Rectangle((5.5, -0.5), 1, 6, color="gray", alpha=0.35))

plt.tight_layout()
plt.show()

## Putting the toy example together: Convolution -> ReLU -> Pooling

Each filter turns the 9 X 9 picture into a small 3 X 3 summary saying *where* its pattern is.
A fully connected layer can then look at these small maps and decide "X" or "O".
A real CNN stacks several of these blocks, and it **learns** the filter weights during training instead of us writing them.

In [ ]:
stages = [
    ("Filter", None),
    ("Convolution", toy_feature_maps),
    ("ReLU", toy_relu),
    ("Max pool", pooled),
]

fig, axes = plt.subplots(3, 4, figsize=(14, 10), gridspec_kw={"width_ratios": [0.6, 1, 1, 0.6]})

for i, name in enumerate(toy_filters):
    for j, (stage, t) in enumerate(stages):
        if t is None:
            show_grid(axes[i, j], toy_filters[name], name, limit=1)
        else:
            show_grid(axes[i, j], t[0, i].numpy(), stage if i == 0 else "", fmt="{:.2f}", limit=1,
                      annotate=(stage == "Max pool"))

plt.tight_layout()
plt.show()

## Model

In [ ]:
layers = []  # list of layers, turned into nn.Sequential below (one input and one output)

### Add a convolutional layer with 32 filters each is 3X3
![alt text](convlayer.png "convolutional layer 32 filters of 3X3")
### Note that you "lose" some pixels due to the border overlap.   You can manually make the shape of the output be the same (28,28) with `padding=1`

**Note**: Keras put the activation inside the layer (`activation='relu'`).  In PyTorch the ReLU is its own layer.
Also, PyTorch needs to be told how many channels come *in* (1 for grayscale) as well as how many filters go *out*.

In [ ]:
layers.append(nn.Conv2d(1, 32, kernel_size=(3,3)))  # see https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
layers.append(nn.ReLU())

### Apply Max pooling, poolsize = 2X2, stride = also 2X2 as it will default to kernel_size if stride is omitted
![alt text](cnnDesign.png "CNN Design")

In [ ]:
layers.append(nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))) # see https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html

In [ ]:
layers.append(nn.Conv2d(32, 64, kernel_size=(3,3)))  # 32 channels in (from previous layer), 64 filters out
layers.append(nn.ReLU())

In [ ]:
layers.append(nn.MaxPool2d(kernel_size=(2,2), stride=(2,2)))

In [ ]:
layers.append(nn.Conv2d(64, 64, kernel_size=(3,3)))
layers.append(nn.ReLU())

In [ ]:
layers.append(nn.Flatten()) # unwrap rows into one single row.

## Add Fully connected layer:
**Note**: Unlike Keras, PyTorch needs the input size of the first Linear layer.

Image size: 28 -> conv 26 -> pool 13 -> conv 11 -> pool 5 -> conv 3, so there are 64 filters X 3 X 3 = 576 values.

In [ ]:
layers.append(nn.Linear(64 * 3 * 3, 64))   #hidden layer
layers.append(nn.ReLU())

In [ ]:
# output layer 0 to 9
# No softmax here: CrossEntropyLoss applies softmax internally.
layers.append(nn.Linear(64, 10))

In [ ]:
model = nn.Sequential(*layers).to(device)
print(model)

In [ ]:
summary(model, input_size=(1, 1, 28, 28), device=device) #sanity check

## Picture of our network

Each box is one layer in `model`. The numbers under each box are the shape of the data **coming out** of that layer:
`channels X rows X cols` for images, or just the length once it has been flattened.

In [ ]:
def draw_cnn_pipeline(model, input_size=(1, 1, 28, 28)):
    """Draw the model as a row of boxes like the 'Putting it all together' slide."""

    # Run a blank image through the model to find each layer's output shape
    shapes = []
    x = torch.zeros(input_size, device=device)
    model.eval()
    with torch.no_grad():
        for layer in model:
            x = layer(x)
            shapes.append(tuple(x.shape[1:]))

    names = [type(layer).__name__ for layer in model]
    n = len(names) + 2      # + input and output boxes

    fig, ax = plt.subplots(figsize=(1.25 * n, 4.2))

    def box(i, label, shape_text, color):
        ax.add_patch(patches.FancyBboxPatch((i + 0.08, 0), 0.84, 2.4, boxstyle="round,pad=0.02",
                                            facecolor=color, alpha=0.25, edgecolor=color, linewidth=2))
        ax.text(i + 0.5, 1.2, label, rotation=90, ha="center", va="center", fontsize=12)
        ax.text(i + 0.5, -0.25, shape_text, ha="center", va="top", fontsize=9, color="dimgray")

    box(0, "Input image", "x".join(map(str, input_size[1:])), "gray")

    for i, (name, shape) in enumerate(zip(names, shapes), start=1):
        label = {"Conv2d": "Convolution", "MaxPool2d": "Pooling", "Linear": "Fully connected"}.get(name, name)
        box(i, label, "x".join(map(str, shape)), LAYER_COLORS.get(name, "gray"))

    box(n - 1, "Digit 0 - 9", "softmax", "gray")

    ax.annotate("", xy=(n, -0.8), xytext=(0, -0.8),
                arrowprops=dict(arrowstyle="->", color="gray", linewidth=2))
    ax.set_xlim(-0.1, n + 0.1)
    ax.set_ylim(-1.0, 2.6)
    ax.axis("off")
    ax.set_title("Data flows left to right", fontsize=14)
    plt.show()


draw_cnn_pipeline(model)

In [ ]:
##### SIMPLEr MODEL FOR VISUALIZATION

#model = nn.Sequential(
#    nn.Conv2d(1, 16, (3, 3)), nn.ReLU(),   # 16 filters in the first Conv layer
#    nn.MaxPool2d((2, 2)),                  # MaxPooling layer to reduce dimensions
#    nn.Conv2d(16, 32, (3, 3)), nn.ReLU(),  # 32 filters in the second Conv layer
#    nn.MaxPool2d((2, 2)),                  # Another MaxPooling layer
#    nn.Flatten(),                          # Flatten the feature maps
#    nn.Linear(32 * 5 * 5, 10)              # Dense layer for 10-class classification (MNIST)
#).to(device)

## Train

In [ ]:
# ---------------------------------------------------------
# Loss function and optimizer
# ---------------------------------------------------------

loss_function = nn.CrossEntropyLoss()

# Keras 'rmsprop' default learning rate is 0.001
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)

In [ ]:
# ---------------------------------------------------------
# Evaluate: returns (loss, accuracy) like Keras model.evaluate()
# ---------------------------------------------------------

def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch_images, batch_labels in loader:

            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            predictions = model(batch_images)

            total_loss += loss_function(predictions, batch_labels).item() * len(batch_labels)
            correct += (predictions.argmax(dim=1) == batch_labels).sum().item()
            total += len(batch_labels)

    return total_loss / total, correct / total

In [ ]:
# ---------------------------------------------------------
# Train the network
# Keras model.fit() did this loop for us.  Here we write it out,
# and keep a history dictionary to plot afterwards.
# ---------------------------------------------------------

epochs = 10

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch_images, batch_labels in train_loader:

        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        # Clear gradients from previous iteration
        optimizer.zero_grad()

        # Forward pass
        predictions = model(batch_images)

        # Calculate loss
        loss = loss_function(predictions, batch_labels)

        # Calculate gradients
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item() * len(batch_labels)
        correct += (predictions.argmax(dim=1) == batch_labels).sum().item()
        total += len(batch_labels)

    val_loss, val_accuracy = evaluate(model, test_loader)

    history["loss"].append(total_loss / total)
    history["accuracy"].append(correct / total)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{epochs}  "
        f"loss: {history['loss'][-1]:.4f}  accuracy: {history['accuracy'][-1]:.4f}  "
        f"val_loss: {val_loss:.4f}  val_accuracy: {val_accuracy:.4f}"
    )

In [ ]:
# Plot training & validation accuracy values
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history['loss'])
plt.plot(history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
evaluate(model, test_loader)  # (loss, accuracy)

In [ ]:
torch.save(model, 'mycnnmodel.pt')  # a file should appear (saves the whole model, layers + weights)

In [ ]:
# weights_only=False is needed to load a whole model (not just a dictionary of weights)
modelReadIn = torch.load('mycnnmodel.pt', weights_only=False)

In [ ]:
# Run all test images through the model, 1000 at a time to save memory
modelReadIn.eval()

with torch.no_grad():
    predictions = torch.cat([
        torch.softmax(modelReadIn(batch.to(device)), dim=1).cpu()
        for batch in torch.split(test_images, 1000)
    ]).numpy()

In [ ]:
print( np.argmax(predictions[10]))   # just pick some random images
plt.imshow(printable_test_images[10])

# Putting it all together on a real digit

Here is one test image going through **every** layer of the trained model, left to right.
For each convolution block we show the 4 channels that reacted most strongly.
After `Flatten` the image is just a long column of numbers, and the fully connected layers turn it into
10 probabilities, one per digit.

In [ ]:
def show_digit_through_network(model, image_index, n_channels=4):

    image = test_images[image_index].unsqueeze(0).to(device)

    # Save the output of every layer
    outputs = []
    model.eval()
    with torch.no_grad():
        x = image
        for layer in model:
            x = layer(x)
            outputs.append(x[0].cpu().numpy())

    names = [type(layer).__name__ for layer in model]
    probabilities = torch.softmax(torch.tensor(outputs[-1]), dim=0).numpy()

    image_stages = [i for i, o in enumerate(outputs) if o.ndim == 3]          # conv / relu / pool
    vector_stages = [i for i, o in enumerate(outputs[:-1]) if o.ndim == 1]   # flatten / linear / relu

    # For each conv block, pick the channels with the strongest ReLU output
    chosen = {}
    for i in image_stages:
        if names[i] == "Conv2d":
            strength = outputs[i + 1].mean(axis=(1, 2))
            current = np.argsort(strength)[::-1][:n_channels]
        chosen[i] = current

    n_cols = 1 + len(image_stages) + len(vector_stages) + 1
    widths = [1.3] + [1] * len(image_stages) + [0.35] * len(vector_stages) + [3]
    fig = plt.figure(figsize=(sum(widths) * 1.6, 8))
    grid = fig.add_gridspec(n_channels, n_cols, width_ratios=widths, wspace=0.4)

    def header(ax, name, shape):
        ax.set_title(f"{name}\n{shape}", fontsize=9, color="black",
                     bbox=dict(facecolor=LAYER_COLORS.get(name, "lightgray"), alpha=0.25,
                               edgecolor=LAYER_COLORS.get(name, "lightgray")))

    # Input
    ax = fig.add_subplot(grid[1:3, 0])
    ax.imshow(printable_test_images[image_index], cmap="gray")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("Input\n1x28x28", fontsize=9)

    # Image-shaped layers
    for col, i in enumerate(image_stages, start=1):
        maps = outputs[i]
        for row, ch in enumerate(chosen[i]):
            ax = fig.add_subplot(grid[row, col])
            limit = np.abs(maps[ch]).max() or 1
            ax.imshow(maps[ch], cmap=signed_cmap, vmin=-limit, vmax=limit)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 1 or names[i] == "Conv2d":
                ax.set_ylabel(f"ch {ch}", fontsize=8)
            if row == 0:
                header(ax, names[i], "x".join(map(str, maps.shape)))

    # Flattened layers, drawn as a tall strip
    for col, i in enumerate(vector_stages, start=1 + len(image_stages)):
        ax = fig.add_subplot(grid[:, col])
        v = outputs[i].reshape(-1, 1)
        limit = np.abs(v).max() or 1
        ax.imshow(v, cmap=signed_cmap, vmin=-limit, vmax=limit, aspect="auto")
        ax.set_xticks([]); ax.set_yticks([])
        header(ax, names[i], str(len(v)))

    # Output probabilities
    ax = fig.add_subplot(grid[:, -1])
    predicted = probabilities.argmax()
    colors = ["#86b6ef"] * 10
    colors[predicted] = "#1c5cab"
    ax.barh(range(10), probabilities, color=colors, height=0.7)
    ax.set_yticks(range(10))
    ax.invert_yaxis()
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("probability")
    ax.spines[["top", "right"]].set_visible(False)
    ax.text(probabilities[predicted] + 0.02, predicted, f"{probabilities[predicted]:.2f}",
            va="center", fontsize=10)
    ax.set_title(f"Output: predicts {predicted}", fontsize=11)

    plt.show()


show_digit_through_network(model, 10)

In [ ]:
# Try other images.  Notice the maps get smaller and more abstract as we go right.
show_digit_through_network(model, 12)

# CNN poster: one digit, start to finish

One big figure that tells the whole story using the **trained** model:

1. **Input** -> 2. **learned filters** -> 3. **feature maps** -> 4. **ReLU** -> 5. **pooling**
* **Zoom in:** how one filter slides over the image to make one feature map (and keeps *where* things are)
* **Position matters:** the same layer-2 channel on two different digits; the red **+** marks the center of its response
* **Deeper layers:** a layer-2 channel mixes several layer-1 maps, and the fully connected layers turn it all into probabilities

**Where do the names come from?** The network never names anything. We add the labels ourselves:
* Layer-1 filters are compared with simple hand-made templates (vertical edge, horizontal edge, ...). If none match well the filter is called a *mixed pattern*.
* Layer-2 channels are labeled with the digit they respond to most, averaged over 1000 test images.

The filters are learned from random starting weights, so the picture (and the labels) change every time you retrain.

**Colors:** filters and raw convolution output use red = positive, blue = negative.
After ReLU nothing is negative, so those maps use dark purple = 0 (nothing found) up to yellow = strong response.

In [ ]:
# ---------------------------------------------------------
# Helpers for the poster
# ---------------------------------------------------------
from matplotlib.patches import ConnectionPatch
from matplotlib.colors import to_rgba


def layer_outputs(model, images):
    """Run images (N, 1, 28, 28) through the model and return every layer's output (on the CPU)."""
    outputs = []
    model.eval()
    with torch.no_grad():
        x = images.to(device)
        for layer in model:
            x = layer(x)
            outputs.append(x.cpu())
    return outputs


# Hand-made 3 X 3 patterns used only to put a NAME on each learned filter
FILTER_TEMPLATES = {
    "vertical edge":   [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    "horizontal edge": [[-1, -1, -1], [0, 0, 0], [1, 1, 1]],
    "diagonal edge \\": [[0, 1, 1], [-1, 0, 1], [-1, -1, 0]],
    "diagonal edge /": [[1, 1, 0], [1, 0, -1], [0, -1, -1]],
    "vertical line":   [[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]],
    "horizontal line": [[-1, -1, -1], [2, 2, 2], [-1, -1, -1]],
    "spot":            [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]],
}


def describe_filter(weights, min_match=0.6):
    """Name a 3 X 3 filter after the template it looks most like (ignoring brightness and sign)."""
    w = np.asarray(weights, dtype=float)
    w = w - w.mean()
    best_name, best_score = "mixed pattern", min_match

    for name, template in FILTER_TEMPLATES.items():
        t = np.asarray(template, dtype=float)
        t = t - t.mean()
        score = abs((w * t).sum()) / (np.linalg.norm(w) * np.linalg.norm(t) + 1e-8)
        if score > best_score:
            best_name, best_score = name, score

    return best_name


def center_row(feature_map):
    """Row of the 'center of mass' of a feature map (where the response is, top to bottom)."""
    rows = torch.arange(feature_map.shape[0], dtype=torch.float32)
    return float((feature_map.sum(1) * rows).sum() / (feature_map.sum() + 1e-8))


def center_col(feature_map):
    return center_row(feature_map.T)


def style_panel(subfig, title, color):
    subfig.set_facecolor(to_rgba(color, 0.10))
    subfig.patch.set_edgecolor(color)
    subfig.patch.set_linewidth(2)
    subfig.suptitle(title, fontsize=13, fontweight="bold")


def bare(ax, label=None):
    ax.set_xticks([])
    ax.set_yticks([])
    if label:
        ax.set_xlabel(label, fontsize=8)

In [ ]:
def cnn_poster(model, image_index=None, compare_index=None, n_sample=1000):
    """Draw the whole CNN story for one test image.

    image_index:   test image to explain (default: the first correctly predicted 9)
    compare_index: second image for 'position matters' (default: the first correctly predicted 6)
    """
    convs = [i for i, layer in enumerate(model) if isinstance(layer, nn.Conv2d)]
    c1, c2 = convs[0], convs[1]          # each conv is followed by ReLU (+1) and MaxPool (+2)

    # ---- Look at a sample of test images: pick examples and learn what each layer-2 channel likes
    sample_outputs = layer_outputs(model, test_images[:n_sample])
    sample_labels = test_labels[:n_sample]
    sample_predicted = sample_outputs[-1].argmax(1)

    if image_index is None:
        image_index = ((sample_labels == 9) & (sample_predicted == 9)).nonzero()[0].item()
    if compare_index is None:
        compare_index = ((sample_labels == 6) & (sample_predicted == 6)).nonzero()[0].item()

    relu2_strength = sample_outputs[c2 + 1].mean((2, 3))                    # (n_sample, channels)
    profile = torch.stack([relu2_strength[sample_labels == d].mean(0) for d in range(10)])
    favorite = (profile / (profile.mean(0) + 1e-8)).argmax(0)            # digit each channel likes most
    class_maps = torch.stack([sample_outputs[c2 + 1][sample_labels == d].mean(0) for d in range(10)])
    del sample_outputs

    # ---- Run our two images through the network
    pair = layer_outputs(model, test_images[[image_index, compare_index]])
    digit_a = test_labels[image_index].item()
    digit_b = test_labels[compare_index].item()
    image = printable_test_images[image_index]

    conv1_out, relu1_out, pool1_out = pair[c1][0], pair[c1 + 1][0], pair[c1 + 2][0]
    relu2_out = pair[c2 + 1]                       # (2, channels, rows, cols): image a and image b
    probabilities = torch.softmax(pair[-1][0], dim=0).numpy()
    predicted = probabilities.argmax()

    w1 = model[c1].weight.detach().cpu()[:, 0]     # (32, 3, 3)
    w2 = model[c2].weight.detach().cpu()           # (64, 32, 3, 3)
    names = [describe_filter(w) for w in w1]

    # ---- Pick 8 layer-1 filters: the best example of each template, then the most active others
    chosen = []
    for name in FILTER_TEMPLATES:
        matches = [i for i in range(len(w1)) if names[i] == name and i not in chosen]
        if matches:
            chosen.append(max(matches, key=lambda i: float(relu1_out[i].mean())))
    contrast = relu1_out.amax((1, 2)) - relu1_out.flatten(1).median(1).values   # bright digit, dark background
    by_contrast = contrast.argsort(descending=True).tolist()
    chosen = (chosen + [i for i in by_contrast if i not in chosen])[:8]

    # Filter for the zoom-in: the named filter with the strongest response
    named = [i for i in chosen if names[i] != "mixed pattern"] or chosen
    zoom = max(named, key=lambda i: float(relu1_out[i].max()))
    r, c = np.unravel_index(conv1_out[zoom].argmax().item(), conv1_out[zoom].shape)

    # Layer-2 channel for 'position matters': its response sits lower for digit b than for digit a,
    # both on average over all those digits AND on our two images, and it responds strongly to both
    map_a, map_b = class_maps[digit_a], class_maps[digit_b]          # (channels, rows, cols)
    strength = torch.minimum(map_a.amax((1, 2)), map_b.amax((1, 2)))

    def position_score(ch):
        shift_average = center_row(map_b[ch]) - center_row(map_a[ch])
        shift_ours = center_row(relu2_out[1, ch]) - center_row(relu2_out[0, ch])
        return min(shift_average, shift_ours) * float(strength[ch])

    k = max(range(len(strength)), key=position_score)

    # ======================================================
    # Layout
    # ======================================================
    fig = plt.figure(figsize=(24, 17), layout="constrained")
    fig.suptitle(f"Convolutional Neural Network Example - Digit {digit_a}\n"
                 "pixels -> simple features -> feature maps (where) -> combinations -> classification",
                 fontsize=18, fontweight="bold")

    row1, row2, row3 = fig.subfigures(3, 1, height_ratios=[1.05, 1, 1], hspace=0.02)

    # ---------------- Row 1: the pipeline ----------------
    p_in, p_filt, p_conv, p_relu, p_pool = row1.subfigures(1, 5, width_ratios=[0.9, 2, 2, 2, 2], wspace=0.01)

    style_panel(p_in, "1. Input image", "gray")
    ax = p_in.subplots()
    ax.imshow(image, cmap="gray")
    bare(ax, f"28 x 28 pixels (grayscale)\nlabel: {digit_a}")

    panels = [
        (p_filt, "2. Convolution filters (learned)", LAYER_COLORS["Conv2d"]),
        (p_conv, "3. Feature maps (after convolution)", LAYER_COLORS["Conv2d"]),
        (p_relu, "4. ReLU (keep positive values)", LAYER_COLORS["ReLU"]),
        (p_pool, "5. Pooling (2 x 2 max)", LAYER_COLORS["MaxPool2d"]),
    ]
    for p, title, color in panels:
        style_panel(p, title, color)

    filt_axes = p_filt.subplots(2, 4).flatten()
    conv_axes = p_conv.subplots(2, 4).flatten()
    relu_axes = p_relu.subplots(2, 4).flatten()
    pool_axes = p_pool.subplots(2, 4).flatten()

    for n, i in enumerate(chosen):
        label = f"#{i} {names[i]}"
        show_grid(filt_axes[n], w1[i].numpy(), annotate=False)
        bare(filt_axes[n], label)

        limit = conv1_out[i].abs().max().item() or 1
        conv_axes[n].imshow(conv1_out[i], cmap=signed_cmap, vmin=-limit, vmax=limit)
        bare(conv_axes[n], label)

        relu_axes[n].imshow(relu1_out[i], cmap="viridis", vmin=0)
        bare(relu_axes[n], label)

        pool_axes[n].imshow(pool1_out[i], cmap="viridis", vmin=0)
        bare(pool_axes[n], label)

    # ---------------- Row 2: zoom in + position matters ----------------
    p_zoom, p_pos = row2.subfigures(1, 2, width_ratios=[2.3, 1], wspace=0.01)

    style_panel(p_zoom, f"Zoom in: how one filter (#{zoom}, {names[zoom]}) produces a feature map and keeps location",
                LAYER_COLORS["Conv2d"])
    z_in, z_filt, z_slide, z_map = p_zoom.subplots(1, 4, width_ratios=[1, 0.7, 1, 1.15])

    z_in.imshow(image, cmap="gray")
    z_in.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 3, 3, fill=False, edgecolor="red", linewidth=2.5))
    z_in.set_title("Input")
    bare(z_in)

    show_grid(z_filt, w1[zoom].numpy(), "Filter (3 x 3 weights)", fmt="{:.2f}")
    for y_in, y_filt in [(r - 0.5, -0.5), (r + 2.5, 2.5)]:
        z_filt.add_artist(ConnectionPatch(xyA=(c + 2.5, y_in), coordsA=z_in.transData,
                                          xyB=(-0.5, y_filt), coordsB=z_filt.transData,
                                          color="red", linestyle="--"))

    z_slide.imshow(image, cmap="gray", alpha=0.55)
    z_slide.set_xticks(np.arange(-0.5, 28, 1), minor=True)
    z_slide.set_yticks(np.arange(-0.5, 28, 1), minor=True)
    z_slide.grid(which="minor", color="lightgray", linewidth=0.4)
    z_slide.tick_params(which="both", length=0)
    z_slide.set_xticks([]); z_slide.set_yticks([])
    z_slide.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 3, 3, fill=False, edgecolor="red", linewidth=2.5))
    z_slide.set_title("Convolve: slide across the image")
    z_slide.set_xlabel("the filter is applied at every position", fontsize=9)

    im = z_map.imshow(relu1_out[zoom], cmap="viridis", vmin=0)
    z_map.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, edgecolor="red", linewidth=2.5))
    z_map.set_title("Feature map (26 x 26, after ReLU)")
    bare(z_map, "red square = the value from the red patch")
    bar = p_zoom.colorbar(im, ax=z_map, ticks=[0, relu1_out[zoom].max().item()], shrink=0.8)
    bar.ax.set_yticklabels(["Low\n(not found)", "High\n(found)"])

    style_panel(p_pos, f"Position matters: layer-2 channel #{k}", "#eda100")
    pos_axes = p_pos.subplots(1, 2)
    vmax = relu2_out[:, k].max().item()
    for j, (ax, idx, digit) in enumerate(zip(pos_axes, [image_index, compare_index], [digit_a, digit_b])):
        fmap = relu2_out[j, k]
        size = fmap.shape[0]
        ax.imshow(fmap, cmap="viridis", vmin=0, vmax=vmax, interpolation="bilinear",
                  extent=(-0.5, 27.5, 27.5, -0.5))
        ax.contour(printable_test_images[idx], levels=[100], colors="white", linewidths=0.8)
        # center of the response, scaled from the 11 X 11 map up to 28 X 28
        y = (center_row(fmap) + 0.5) * 28 / size - 0.5
        x = (center_col(fmap) + 0.5) * 28 / size - 0.5
        ax.plot(x, y, "+", color="red", markersize=18, markeredgewidth=3)
        ax.set_title(f"Feature map for a {digit}")
        bare(ax, "white outline = the digit")

    # ---------------- Row 3: deeper layers + classification ----------------
    p_deep, p_fc = row3.subfigures(1, 2, width_ratios=[3, 1], wspace=0.01)
    style_panel(p_deep, "Deeper layers combine features and their relative positions", LAYER_COLORS["Linear"])
    style_panel(p_fc, "Fully connected layers (classification)", LAYER_COLORS["Linear"])

    d_in, d_filt, d_out = p_deep.subfigures(1, 3, width_ratios=[1.15, 1.15, 2])

    pool_contrast = pool1_out.amax((1, 2)) - pool1_out.flatten(1).median(1).values
    contribution = w2[k].abs().sum((1, 2)) * pool_contrast
    inputs = contribution.argsort(descending=True)[:3].tolist()

    d_in.suptitle("First-layer feature maps\n(the 3 that feed channel #%d most)" % k, fontsize=11)
    for ax, ch in zip(d_in.subplots(1, 3), inputs):
        ax.imshow(pool1_out[ch], cmap="viridis", vmin=0)
        bare(ax, f"#{ch}\n{names[ch]}")

    d_filt.suptitle(f"Second-layer filter #{k}\n(its 3 x 3 weights for those maps)", fontsize=11)
    for ax, ch in zip(d_filt.subplots(1, 3), inputs):
        show_grid(ax, w2[k, ch].numpy(), annotate=False)
        bare(ax, f"weights for #{ch}")
    d_filt.supxlabel("channel #%d adds up all 32 of these products" % k, fontsize=9)

    others = [ch for ch in relu2_out[0].mean((1, 2)).argsort(descending=True).tolist() if ch != k][:3]
    d_out.suptitle("Second-layer feature maps (labeled with the digit each channel likes most)", fontsize=11)
    for ax, ch in zip(d_out.subplots(1, 4), [k] + others):
        ax.imshow(relu2_out[0, ch], cmap="viridis", vmin=0, interpolation="bilinear")
        bare(ax, f"#{ch}: {int(favorite[ch])}-like pattern")
        if ch == k:
            for spine in ax.spines.values():
                spine.set_edgecolor("red")
                spine.set_linewidth(2.5)

    fc_text, fc_image = p_fc.subplots(1, 2, width_ratios=[1, 1])
    fc_text.axis("off")
    fc_text.text(0.05, 0.97, "Output probabilities:", fontsize=12, va="top")
    for d, p in enumerate(probabilities):
        fc_text.text(0.15, 0.87 - d * 0.085, f"{d}: {p:.2f}", family="monospace", fontsize=12, va="top",
                     fontweight="bold" if d == predicted else "normal",
                     bbox=dict(facecolor="#f4b6c2", edgecolor="none") if d == predicted else None)

    fc_image.imshow(image, cmap="gray_r")
    bare(fc_image)
    fc_image.set_xlabel(f"Predicted: {predicted}", fontsize=16, fontweight="bold")

    plt.show()


cnn_poster(model)

In [ ]:
# Try it on other images.  Any two test image numbers work:
# the first is explained, the second is used for 'position matters'.
# (test image 4 is a 4, test image 8 is a 5)
cnn_poster(model, image_index=4, compare_index=8)

In [ ]:
summary(model, input_size=(1, 1, 28, 28), device=device)

# Visualizing filters
## model is a list of layers. Because ReLU is its own layer in PyTorch, the convolutional layers are at index 0, 3, 6

In [ ]:
all_layers = list(model)

In [ ]:
print ( all_layers ) #more helpful than Keras

In [ ]:
print ( all_layers[0] ) #verify layer 0 in a Conv2d

In [ ]:
print ( all_layers[1] ) # note that layer 1 is a ReLU, layer 2 is a MaxPool, etc.

In [ ]:
# filters is all of the filters/weights in the first conv layer
# .detach() = no gradients, .cpu() = off the GPU, .numpy() = numpy array
filters = all_layers[0].weight.detach().cpu().numpy()
biases = all_layers[0].bias.detach().cpu().numpy()

In [ ]:
#print (filters )

In [ ]:
# first layer has 32 X 1 X 3 X 3:  32 filters, 1 is for monochrome (as opposed to RGB), rows and cols are 3
# (Keras stored this as 3 X 3 X 1 X 32, so the filter number is now FIRST)
print ('conv layer 0', filters.shape)

In [ ]:
# FIRST CONVOLUTIONAL LAYER:  look for features
# Likely will have edges ( horizaontal, vertical, diagonal):  Light on one side, dark on other side
#


# Create figure for subplots
figure1 = plt.figure(figsize=(8, 12))
columns = 6
rows = 6
n_filters = 32  # Number of filters in the layer (adjust as necessary)




# Loop over each filter and plot it
for i in range(1, n_filters + 1):
    f = filters[i - 1, :, :, :]  # Extract the i-th filter

    ax = plt.subplot(rows, columns, i)  # Add subplot in a grid
    ax.set_xticks([])  # Remove x-axis ticks
    ax.set_yticks([])  # Remove y-axis ticks

    plt.imshow(f[0, :, :], cmap='gray')  # Display the filter (assuming grayscale)

# Show the plot with all filters
plt.show()

# white is a high value in filter,  black is low

## What each learned filter does to a digit

Top row: 8 of the 32 filters the network **learned** in the first convolutional layer
(compare them with the hand-made `\` and `/` filters in the toy example).
Bottom row: the feature map each filter produces from the same digit, after ReLU.
Red areas are where the filter found its pattern.

In [ ]:
image_index = 12
image = test_images[image_index].unsqueeze(0).to(device)

conv1 = model[0]
with torch.no_grad():
    maps = torch.relu(conv1(image))[0].cpu().numpy()     # (32, 26, 26)

weights = conv1.weight.detach().cpu().numpy()[:, 0]      # (32, 3, 3)

n_show = 8
fig, axes = plt.subplots(2, n_show + 1, figsize=(2 * (n_show + 1), 4.6))

axes[0, 0].imshow(printable_test_images[image_index], cmap="gray")
axes[0, 0].set_title("Input")
axes[1, 0].axis("off")
axes[0, 0].set_xticks([]); axes[0, 0].set_yticks([])

for i in range(n_show):
    show_grid(axes[0, i + 1], weights[i], f"filter {i}", fmt="{:.1f}")
    limit = maps[i].max() or 1
    axes[1, i + 1].imshow(maps[i], cmap=signed_cmap, vmin=-limit, vmax=limit)
    axes[1, i + 1].set_xticks([]); axes[1, i + 1].set_yticks([])

axes[1, 1].set_ylabel("feature map\n(after ReLU)")
plt.tight_layout()
plt.show()

In [ ]:
#Visualize second layer
#  (e.g., combining edges, textures, or shapes).
#Possible Features Detected by These Filters:
#Curved Shapes: Many of the filters likely detect curves or arcs, which could be parts of digits like "6", "9", or "8", where circular patterns are common.
#Diagonal or Intersection Points: Some filters could be focusing on intersections or diagonal lines, which are important for identifying digits like "4", "7", or "2".
#Textures and Partials: As the filters combine lower-level edges, they may start identifying partial shapes like the bottom loop of a "6" or the top of an "8".



filters = model[3].weight.detach().cpu().numpy()   # second conv layer: 64 X 32 X 3 X 3

# Number of filters in the second convolutional layer
n_filters = filters.shape[0]

# Dynamically determine the number of rows and columns based on the number of filters
columns = 6
rows = (n_filters + columns - 1) // columns  # Ensures enough rows for all filters

# Create figure
figure2 = plt.figure(figsize=(8, 12))

# Loop to plot each filter
for i in range(1, n_filters + 1):
    f = filters[i - 1, :, :, :]  # Extract the i-th filter

    ax = plt.subplot(rows, columns, i)
    ax.set_xticks([])  # Remove x-axis ticks
    ax.set_yticks([])  # Remove y-axis ticks

    plt.imshow(f[0, :, :], cmap='gray')  # Display the filter (first input channel only)

plt.show()

# Feature maps
Keras built a second `Model` whose outputs were the inner layers.
In PyTorch we just pass the image through the layers one at a time and keep the outputs we want.

**Note**: Keras' conv layer output already had ReLU applied, so we keep the output *after* the ReLU.

In [ ]:
# ---------------------------------------------------------
# Run one image through the model, saving the output of
# every layer whose type is in keep_types
# ---------------------------------------------------------

def get_feature_maps(model, image, keep_types=(nn.ReLU,), stop_at=nn.Flatten):

    model.eval()

    outputs = []
    x = image.to(device)

    with torch.no_grad():

        for layer in model:

            if isinstance(layer, stop_at):
                break

            x = layer(x)

            if isinstance(layer, keep_types):
                outputs.append(x.cpu().numpy())

    return outputs

In [ ]:
# visualization of numbers passed through first filter layer

# First layer will show basic features like edges and simple textures.  Some feture maps show the upper loop of a "9" and others show the contrast of edges

# Second layer will show more complex patterns:   Combinations of



# Outputs of the convolutional layers only (conv + ReLU)
testImageIndexToTry = 12
testImage = test_images[testImageIndexToTry].unsqueeze(0)  # add batch dimension: (1, 1, 28, 28)

# Get feature maps for the test image
feature_maps = get_feature_maps(model, testImage)

# Visualize the feature maps from each convolutional layer
for layer_index, feature_map in enumerate(feature_maps):
    num_filters = feature_map.shape[1]   # channels are dimension 1 in PyTorch

    # Dynamically determine the number of rows and columns for the subplots
    rows = int(np.sqrt(num_filters))  # Number of rows
    columns = int(np.ceil(num_filters / rows))  # Number of columns

    fig = plt.figure(figsize=(15, 15))
    fig.suptitle(f'Feature Maps from Layer {layer_index + 1}', fontsize=16)

    for i in range(1, num_filters + 1):
        plt.subplot(rows, columns, i)
        plt.xticks([])  # remove x-axis ticks
        plt.yticks([])  # remove y-axis ticks
        plt.imshow(feature_map[0, i - 1, :, :], cmap='gray')

    plt.tight_layout()  # Adjust subplots to fit in figure area.
    plt.show()

In [ ]:
# put an image through the model to see the output
columns = 6
rows = 6
testImageIndexToTry = 12
testImage = test_images[testImageIndexToTry]
testImage = testImage.unsqueeze(0)

plt.imshow(printable_test_images[testImageIndexToTry], cmap = 'gray')

feature_output = get_feature_maps(model, testImage)
for the_filter in feature_output:
    figure2 = plt.figure(figsize=(28,28))
    for i in range ( 1, 32 + 1):
        figure2 = plt.subplot(rows,columns,i)
        figure2.set_xticks([])
        figure2.set_yticks([])
        plt.imshow(the_filter[0, i-1, :, :], cmap='gray') # cmap=binary,   or hot or gray ,
    plt.show()

    #note first filter  is detecting top and bottom edges

# example index numbers
# 0:  3,10,13
# 1:  2,5
# 2:  1,
# 6:  11,
# 4:  4,6
# 9: 7,9,12


    # I cannot tell what index 8 is

In [ ]:
# Get filters for the first conv layer
filters = model[0].weight.detach().cpu().numpy()

# Normalize filter values to [0, 1] for better visualization
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)

n_filters = filters.shape[0]
rows = 4  # Adjust according to the number of filters
columns = n_filters // rows

# Visualize the filters
for i in range(n_filters):
    f = filters[i, :, :, :]
    plt.subplot(rows, columns, i + 1)
    plt.xticks([])  # remove x-axis ticks
    plt.yticks([])  # remove y-axis ticks
    plt.imshow(f[0, :, :], cmap='gray')  # Show the filter
plt.show()

In [ ]:
# Same as Keras model.layers[:5]:  conv, pool, conv, pool, conv
# (keep the output after each ReLU and each MaxPool)
testImageIndexToTry = 12
testImage = test_images[testImageIndexToTry].unsqueeze(0)

# Get feature maps for the test image
feature_maps = get_feature_maps(model, testImage, keep_types=(nn.ReLU, nn.MaxPool2d))

# Visualize the feature maps from each convolutional layer
for layer_index, feature_map in enumerate(feature_maps):
    num_filters = feature_map.shape[1]

    # Dynamically determine the number of rows and columns for the subplots
    rows = int(np.sqrt(num_filters))  # Number of rows
    columns = int(np.ceil(num_filters / rows))  # Number of columns

    fig = plt.figure(figsize=(20, 15))
    fig.suptitle(f'Feature Maps from Layer {layer_index + 1}', fontsize=16)

    for i in range(1, num_filters + 1):
        plt.subplot(rows, columns, i)
        plt.xticks([])  # remove x-axis ticks
        plt.yticks([])  # remove y-axis ticks
        plt.imshow(feature_map[0, i - 1, :, :], cmap='gray')
    plt.show()